# Notebook BNL

## Tree style file system for BNL folders and files

### BNL old
Original batch of data in `BNL`

```bash
BNL/
│
├── $RECYCLE.BIN/
├── @eaDir/
├── @tmp/
│
├── protected_000/
│   ├── <number_newspaper_alias1_year-month-day_XX>/ or <number_newspaper_alias1_year-month-day>
│   │   ├── <number_newspaper_alias1_year-month-day_XX>-mets.xml
│   │   ├── <number_newspaper_alias1_year-month-day_XX>.pdf
│   │   ├── images/
│   │   │   └── <year-month-day_XX-XXXXX>.tif
│   │   ├── pdf/
│   │   │   └── <year-month-day_XX-XXXXX>.pdf
│   │   └── text/
│   │       └── <year-month-day_XX-XXXXX>.xml  or <year-month-day_XX-XXXXX>-alto.xml  
│   ├── <number_newspaper_alias2_year-month-day_XX>/
│   └── ... many more issues
│
├── ...
├── protected_054/
│
├── public_domain_000/
│   ├── <number_newspaper_alias1_year-month-day_XX>/ or <number_newspaper_alias1_year-month-day>
│   │   ├── <number_newspaper_alias1_year-month-day_XX>-mets.xml
│   │   ├── <number_newspaper_alias1_year-month-day_XX>.pdf
│   │   ├── images/
│   │   │   └── <year-month-day_XX-XXXXX>.tif
│   │   ├── pdf/
│   │   │   └── <year-month-day_XX-XXXXX>.pdf
│   │   └── text/
│   │       └── <year-month-day_XX-XXXXX>.xml or <year-month-day_XX-XXXXX>-alto.xml  
│   ├── <number_newspaper_alias2_year-month-day_XX>/
│   └── ...
│
├── ...
├── public_domain_008/
```


### BNL-new

End 2024 batch of data in `BNL-new/impresso`

Latest batch of data in `BNL-new/2025-impresso-3`

```bash
BNL-new/
│
├── 2025-impresso-3/
│   ├── _arks/ 
│   │   └── impresso_<newspaper_title1>/   (ark:70795/md44c3ks09, ... (ark_id of all issues from this newspaper))
│   │   └── ...
│   │   
│   ├── <alias1>/
│   │   └── <ark_id>/   (multiple folders) (ark_id starts always with 70795_)
│   │        └── <issue_date>/
│   │             ├── <year-month-day_XX>-mets.xml
│   │             ├── <year-month-day_XX>.pdf
│   │             ├── images/ <year-month-day_XX-XXXXX>.tif
│   │             ├── pdf/ <year-month-day_XX-XXXXX>.pdf
│   │             ├── text/ <year-month-day_XX-XXXXX>-alto.xml
│   │             ├── thumbnails/ <year-month-day_XX-XXXXX>.jpg
│   │             └── blackwhiteimages/ <year-month-day_XX-XXXXX>.png (not always)
│   │
│   ├── <alias2>/
│   ├── <alias3>/
│   └── ...
│
└── impresso/
    └── public_domain/
         ├── newspaper/
         │   ├── <alias1>/
         │   │   └── <year>/
         │   │       └── <ark_id without 70795_>/
         │   │            ├── <number_newspaper_alias_year-month-day_XX>-mets.xml
         │   │            ├── <number_newspaper_alias_year-month-day_XX>.pdf
         │   │            ├── images/ <year-month-day_XX-XXXXX>.tif
         │   │            ├── pdf/ <year-month-day_XX-XXXXX>.pdf
         │   │            ├── text/ <year-month-day_XX-XXXXX>.xml
         │   │            ├── blackwhiteimages/ <year-month-day_XX-XXXXX>.png (not always)
         │   │            └── thumbnails/ <year-month-day_XX-XXXXX>.jpg
         │   ├── <alias2>/
         │   └── ...
         │
         └── serial/
             ├── memoriala/
             │   ├── 1814/
             │   ├── ...
             │   └── 1881/
             │       └── <year-month-day_XX>/
             │            ├── <year-month-day_XX>-mets.xml
             │            ├── <year-month-day_XX>.pdf             
             │            ├── images/ <year-month-day_XX-XXXXX>.tif
             │            ├── pdf/ <year-month-day_XX-XXXXX>.pdf
             │            ├── text/ <year-month-day_XX-XXXXX>-alto.xml 
             │            ├── blackwhiteimages/ <year-month-day_XX-XXXXX>.png (not always)
             │            └── thumbnails/ <year-month-day_XX-XXXXX>.jpg
             ├── memorialarlon/ same as memoriala/
             └── socbotmem/ 
             │    ├── <year>/
             │       └── <ark_id without 70795_>/
             │            ├── <number_serial_alias_year-month-day_XX>-mets.xml
             │            ├── <number_serial_alias_year-month-day_XX>.pdf
             │            ├── images/ <year-month-day_XX-XXXXX>.tif
             │            ├── pdf/ <year-month-day_XX-XXXXX>.pdf
             │            ├── text/ <year-month-day_XX-XXXXX>.xml
             │            ├── blackwhiteimages/ <year-month-day_XX-XXXXX>.png (not always)
             │            └── thumbnails/ <year-month-day_XX-XXXXX>.jpg
```

## Paths to each issue

### Imports

In [1]:
import json
import os
import requests
import pandas as pd
import numpy as np
import certifi
import string
import unicodedata
from datetime import datetime, date, timedelta
from bs4 import BeautifulSoup
import unicodedata
from collections import OrderedDict
from collections import defaultdict
import re
import sys

In [5]:
BASE_PATH = '/mnt/project_impresso/original'
OLD_BNL_PATH = os.path.join(BASE_PATH, 'BNL/')
NEW_BNL_PATH = os.path.join(BASE_PATH, 'BNL-new/impresso/public_domain/')
LATEST_BNL_PATH = os.path.join(BASE_PATH, 'BNL-new/2025-impresso-3/')

### Functions

In [6]:
def strip_impresso_root(path: str) -> str:
    return path[len(BASE_PATH):] if path.startswith(BASE_PATH) else path


In [7]:
path = "/mnt/project_impresso/original/BNL/protected_042/2281485_newspaper_obermosel_1946-05-27_01/2281485_newspaper_obermosel_1946-05-27_01-mets.xml"

clean_path = strip_impresso_root(path)
clean_path


'/BNL/protected_042/2281485_newspaper_obermosel_1946-05-27_01/2281485_newspaper_obermosel_1946-05-27_01-mets.xml'

In [109]:
def list_to_nested_json_sorted(entries_list, include_to_ingest=False):
    """
    Convert a flat list of issue dicts into:
    alias → year → month → list of day-sorted entries.

    If include_to_ingest=True, include the 'to_ingest' field.
    """

    # Step 1: Build nested structure
    nested = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

    for entry in entries_list:
        alias = entry["alias"]
        date = entry["date"]                  # "YYYY-MM-DD"
        year, month, day = date.split("-")

        item = {
            "day": day,
            "edition": entry["edition"],
            "local_path": entry["local_path"],
        }

        if include_to_ingest:
            item["to_ingest"] = entry["to_ingest"]

        nested[alias][year][month].append(item)

    # Step 2: Sort everything
    output = OrderedDict()

    for alias in sorted(nested.keys()):
        sorted_years = OrderedDict()

        for year in sorted(nested[alias].keys(), key=int):
            sorted_months = OrderedDict()

            for month in sorted(nested[alias][year].keys(), key=int):
                # Sort by DAY (numeric)
                day_sorted_entries = sorted(
                    nested[alias][year][month],
                    key=lambda x: int(x["day"])
                )
                sorted_months[month] = day_sorted_entries

            sorted_years[year] = sorted_months

        output[alias] = sorted_years

    return output


In [101]:
def df_to_nested_json(df):

    result = {}

    for alias, rows in df.groupby("alias"):
        alias_dict = defaultdict(lambda: defaultdict(list))

        for _, r in rows.iterrows():
            # date_temp = r["date"].isoformat()
            year, month, day = r["date"].split("-")
            

            alias_dict[year][month].append({
                "day": day,
                "edition": r["edition"],
                "local_path": r["local_path"]
            })

        result[alias] = alias_dict

    return result


In [102]:
def transform_json_structure(data):

    def nested_dict():
        return defaultdict(nested_dict)

    output = {}

    for alias, entries in data.items():
        new_structure = defaultdict(lambda: defaultdict(list))

        for item in entries:
            date = item["date"]                  # "YYYY-MM-DD"
            local_path = item["local_path"]

            # Extract edition from the local_path suffix (_01, _02...)
            edition = re.search(r"_(\d+)$", local_path)
            edition = edition.group(1) if edition else None

            year, month, day = date.split("-")

            new_structure[year][month].append({
                "day": day,
                "edition": edition,
                "local_path": local_path
            })

        output[alias] = new_structure

    return output



In [3]:
def extract_date_from_mets(filename):
    """
    Extracts a date in the format YYYY-MM-DD from a METS filename.
    Returns the date string or None if not found.
    """
    print("Extracted date from:", filename)
    pattern = r"(\d{4}-\d{2}-\d{2})"
    match = re.search(pattern, filename)
    if not match:
        return None
    date_str = match.group(1)
    
    try:    
        return datetime.strptime(date_str, '%Y-%m-%d').date()
    except ValueError:
        print(f"[ERROR] Invalid date format in filename: {filename}")
        return None


In [ ]:
EDITIONS_MAPPINGS = {1: "a", 2: "b", 3: "c", 4: "d", 5: "e", 6: "f", 7: "g", 8: "h", 9: "i", 10: "j", 11: "k", 12: "l", 13: "m", 14: "n", 15: "o", 16: "p", 17: "q", 18: "r", 19: "s", 20: "t", 21: "u", 22: "v", 23: "w", 24: "x", 25: "y", 26: "z",
                     27: "aa", 28: "ab", 29: "ac", 30: "ad", 31: "ae", 32: "af", 33: "ag", 34: "ah", 35: "ai", 36: "aj", 37: "ak", 38: "al", 39: "am", 40: "an", 41: "ao", 42: "ap", 43: "aq", 44: "ar", 45: "as", 46: "at", 47: "au", 48: "av", 49: "aw", 50: "ax", 51: "ay", 52: "az"
                     }

### Find the local path to each issue for each alias and year/month

In [5]:
df_bnl = pd.DataFrame(columns=['alias', 'date', 'edition', 'local_path'])
rows = []

Function runs around 22 minutes.

In [ ]:
# ---------------------------------------------------------------- ORIGINAL BNL BATCH ---------------------------------------------------------------- #
# find local path, alias and date for issues in OLD_BNL_PATH

mets_count = 0
top_level_folders = os.listdir(OLD_BNL_PATH)

for folder in top_level_folders:
    # Only keep protected_* or public_domain_*
    if not (folder.startswith("protected_") or folder.startswith("public_domain_")):
        continue

    folder_path = os.path.join(OLD_BNL_PATH, folder)
    folders = os.listdir(folder_path)
    # Loop over issue folders
    for issue_folder in folders:
        issue_path = os.path.join(folder_path, issue_folder)

        # Skip non-directories (e.g. .DS_Store)
        if not os.path.isdir(issue_path):
            print(f"this {issue_path}, is not a directory, skipping.")
            continue

        # Look for a file ending with '-mets.xml'
        found_mets = False
        issue_files = os.listdir(issue_path)
        for filename in issue_files:
            if filename.endswith("-mets.xml"):
                # print(f"Found METS file: {os.path.join(issue_path, filename)}")
                mets_count += 1
                found_mets = True
                
                local_path = os.path.join(issue_path, filename)
                
                issue_folder_parts = issue_folder.split('_')
                if len(issue_folder_parts) < 4:
                    print(f"[ERROR] Unexpected path structure: {issue_folder, issue_folder_parts}")
                    continue
                alias = issue_folder_parts[2]
                date = datetime.strptime(issue_folder_parts[3], '%Y-%m-%d').date()
                if len(issue_folder_parts) > 4:
                    edition = EDITIONS_MAPPINGS[int(issue_folder_parts[4])]
                else:
                    edition = 'a'
                rows.append({'alias': alias, 'date': date, 'edition': edition, 'local_path': strip_impresso_root(issue_path)})                
                break  

        
        if not found_mets:
            print(f"[WARNING] No METS file found in: {issue_path}")

print("Total METS files found in OLD BNL batch:", mets_count)


Found METS file: /mnt/project_impresso/original/BNL/protected_042/2281485_newspaper_obermosel_1946-05-27_01/2281485_newspaper_obermosel_1946-05-27_01-mets.xml
Found METS file: /mnt/project_impresso/original/BNL/protected_042/2257740_newspaper_obermosel_1940-04-11_01/2257740_newspaper_obermosel_1940-04-11_01-mets.xml
Found METS file: /mnt/project_impresso/original/BNL/protected_042/2282577_newspaper_obermosel_1946-07-03_01/2282577_newspaper_obermosel_1946-07-03_01-mets.xml
Found METS file: /mnt/project_impresso/original/BNL/protected_042/2280895_newspaper_obermosel_1946-05-04_01/2280895_newspaper_obermosel_1946-05-04_01-mets.xml
Found METS file: /mnt/project_impresso/original/BNL/protected_042/2260969_newspaper_obermosel_1940-08-22_01/2260969_newspaper_obermosel_1940-08-22_01-mets.xml
Found METS file: /mnt/project_impresso/original/BNL/protected_042/2281631_newspaper_obermosel_1946-05-29_01/2281631_newspaper_obermosel_1946-05-29_01-mets.xml
Found METS file: /mnt/project_impresso/origina

This issue folder has an unexpected path structure: /mnt/project_impresso/original/BNL/protected_019/1133553_Newspaper/1133553_Newspaper-mets.xml


In [7]:
rows[:10]

[{'alias': 'obermosel',
  'date': datetime.date(1946, 5, 27),
  'edition': 'a',
  'local_path': '/mnt/project_impresso/original/BNL/protected_042/2281485_newspaper_obermosel_1946-05-27_01'},
 {'alias': 'obermosel',
  'date': datetime.date(1940, 4, 11),
  'edition': 'a',
  'local_path': '/mnt/project_impresso/original/BNL/protected_042/2257740_newspaper_obermosel_1940-04-11_01'},
 {'alias': 'obermosel',
  'date': datetime.date(1946, 7, 3),
  'edition': 'a',
  'local_path': '/mnt/project_impresso/original/BNL/protected_042/2282577_newspaper_obermosel_1946-07-03_01'},
 {'alias': 'obermosel',
  'date': datetime.date(1946, 5, 4),
  'edition': 'a',
  'local_path': '/mnt/project_impresso/original/BNL/protected_042/2280895_newspaper_obermosel_1946-05-04_01'},
 {'alias': 'obermosel',
  'date': datetime.date(1940, 8, 22),
  'edition': 'a',
  'local_path': '/mnt/project_impresso/original/BNL/protected_042/2260969_newspaper_obermosel_1940-08-22_01'},
 {'alias': 'obermosel',
  'date': datetime.date

Number of issues found: 97889 (corresponding to number of mets files found). -1 because of weird mets file name

In [9]:
new_rows = []

Code runs around 25 minutes

In [ ]:
# ---------------------------------------------------------------- NEW BNL BATCH --------------------------------------------------------------------- #
# find local path, alias and date for issues in NEW_BNL_PATH/public_domain/newspaper AND socbotmem is structured the same way "/mnt/project_impresso/original/BNL-new/impresso/public_domain/serial/socbotmem/"
mets_count = 0

# 1) NEWSPAPER + SOCBOTMEM
newspaper_root = os.path.join(NEW_BNL_PATH, 'newspaper')
socbotmem_root = os.path.join(NEW_BNL_PATH, 'serial/socbotmem')
roots_to_walk1 = [newspaper_root, socbotmem_root]

for base in roots_to_walk1:
    for root, dirs, files in os.walk(base):
        found_mets_in_root = False
        for file in files:
            if file.endswith('-mets.xml'):
                found_mets_in_root = True
                mets_count += 1
                
                local_path = os.path.join(root, file)
                
                file_parts = file.split('_')
                if len(file_parts) < 4:
                    print(f"[ERROR] Unexpected file structure: {file_parts}")
                    continue
                date = extract_date_from_mets(file)
                parts = local_path.split(os.sep)
                if len(parts) < 11:
                    print(f"[ERROR] Unexpected path structure: {parts}")
                    continue
                # ark_id = parts[10]
                alias = parts[8]
                last = file_parts[-1].split('-')
                if len(file_parts) > 4 and len(last) > 1:
                    edition = EDITIONS_MAPPINGS[int(last[0])]
                else:
                    edition = 'a'
                new_rows.append({'alias': alias, 'date': date, 'edition': edition, 'local_path': strip_impresso_root(root)})
        if not found_mets_in_root:
            print(f"[WARNING] No METS file found in: {root}")

#. 2) SERIALS (memoriala and memorialarlon)
# find local path, alias and date for issues in NEW_BNL_PATH/public_domain/serial 
memoriala_root = os.path.join(NEW_BNL_PATH, 'serial', 'memoriala')
memorialarlon_root = os.path.join(NEW_BNL_PATH, 'serial', 'memorialarlon')
roots_to_walk2 = [memoriala_root, memorialarlon_root]

for base in roots_to_walk2:
    for root, dirs, files in os.walk(base):
        
        found_mets_in_root = False
        for file in files:
            if file.endswith('-mets.xml'):
                found_mets_in_root = True
                mets_count += 1
                
                local_path = os.path.join(root, file)
                parts = local_path.split(os.sep)
                if len(parts) < 11:
                    print(f"[ERROR] Unexpected path structure: {parts}")
                    continue
                alias = parts[8]
                date = datetime.strptime(parts[10][:10], '%Y-%m-%d').date()
                file_parts = file.split('_')
                last = file_parts[-1].split('-')
                if len(file_parts) > 1 and len(last) > 1:
                    edition = EDITIONS_MAPPINGS[int(last[0])]
                else:
                    edition = 'a'
                print(f"edition: {edition}")
                new_rows.append({'alias': alias, 'date': date, 'edition': edition, 'local_path': strip_impresso_root(root)})
        if not found_mets_in_root:
            print(f"[WARNING] No METS file found in: {root}")
            
print("Total METS files found in NEW BNL batch:", mets_count)

[WARNING] No METS file found in: /mnt/project_impresso/original/BNL-new/impresso/public_domain/newspaper
[WARNING] No METS file found in: /mnt/project_impresso/original/BNL-new/impresso/public_domain/newspaper/luxbauern
[WARNING] No METS file found in: /mnt/project_impresso/original/BNL-new/impresso/public_domain/newspaper/luxbauern/1857
Extracted date from: 3571461_newspaper_luxbauern_1857-06-26_01-mets.xml
[WARNING] No METS file found in: /mnt/project_impresso/original/BNL-new/impresso/public_domain/newspaper/luxbauern/1857/6j6j53kxs/blackwhiteimages
[WARNING] No METS file found in: /mnt/project_impresso/original/BNL-new/impresso/public_domain/newspaper/luxbauern/1857/6j6j53kxs/thumbnails
[WARNING] No METS file found in: /mnt/project_impresso/original/BNL-new/impresso/public_domain/newspaper/luxbauern/1857/6j6j53kxs/pdf
[WARNING] No METS file found in: /mnt/project_impresso/original/BNL-new/impresso/public_domain/newspaper/luxbauern/1857/6j6j53kxs/text
[WARNING] No METS file found in

[WARNING] No METS file found in: /mnt/project_impresso/original/BNL-new/impresso/public_domain/newspaper/luxbauern/1857/6pwvxpr4v/blackwhiteimages
[WARNING] No METS file found in: /mnt/project_impresso/original/BNL-new/impresso/public_domain/newspaper/luxbauern/1857/6pwvxpr4v/thumbnails
[WARNING] No METS file found in: /mnt/project_impresso/original/BNL-new/impresso/public_domain/newspaper/luxbauern/1857/6pwvxpr4v/pdf
[WARNING] No METS file found in: /mnt/project_impresso/original/BNL-new/impresso/public_domain/newspaper/luxbauern/1857/6pwvxpr4v/text
[WARNING] No METS file found in: /mnt/project_impresso/original/BNL-new/impresso/public_domain/newspaper/luxbauern/1857/6pwvxpr4v/images
Extracted date from: 3573225_newspaper_luxbauern_1857-01-23_01-mets.xml
[WARNING] No METS file found in: /mnt/project_impresso/original/BNL-new/impresso/public_domain/newspaper/luxbauern/1857/6nz4648rm/blackwhiteimages
[WARNING] No METS file found in: /mnt/project_impresso/original/BNL-new/impresso/public

In [11]:
new_rows[:10]

[{'alias': 'luxbauern',
  'date': datetime.date(1857, 6, 26),
  'edition': 'a',
  'local_path': '/mnt/project_impresso/original/BNL-new/impresso/public_domain/newspaper/luxbauern/1857/6j6j53kxs'},
 {'alias': 'luxbauern',
  'date': datetime.date(1857, 1, 6),
  'edition': 'a',
  'local_path': '/mnt/project_impresso/original/BNL-new/impresso/public_domain/newspaper/luxbauern/1857/6fkjpht12'},
 {'alias': 'luxbauern',
  'date': datetime.date(1857, 3, 6),
  'edition': 'a',
  'local_path': '/mnt/project_impresso/original/BNL-new/impresso/public_domain/newspaper/luxbauern/1857/6jfdrsj1v'},
 {'alias': 'luxbauern',
  'date': datetime.date(1857, 6, 5),
  'edition': 'a',
  'local_path': '/mnt/project_impresso/original/BNL-new/impresso/public_domain/newspaper/luxbauern/1857/6p174860g'},
 {'alias': 'luxbauern',
  'date': datetime.date(1857, 2, 6),
  'edition': 'a',
  'local_path': '/mnt/project_impresso/original/BNL-new/impresso/public_domain/newspaper/luxbauern/1857/6fwz2rgc2'},
 {'alias': 'luxbaue

In [12]:
len(new_rows)


31815

Number of issues found: 31815 (corresponding to number of mets files found).

In [14]:
latest_rows = []

Code runs 1hour

In [ ]:
# ---------------------------------------------------------------- LATEST BNL BATCH --------------------------------------------------------------------- #
# find local path, alias and date for issues in LATEST_BNL_PATH

mets_count = 0
found_mets_in_root = False
for root, dirs, files in os.walk(LATEST_BNL_PATH):
    for file in files:
        if file.endswith('-mets.xml'):
            found_mets_in_root = True
            mets_count += 1
            local_path = os.path.join(root, file)
            parts = local_path.split(os.sep)
            if len(parts) < 9:
                print(f"[ERROR] Unexpected path structure: {parts}")
                continue
            alias = parts[6]
            date = extract_date_from_mets(file)
            if date is None:
                print(f"[ERROR] Could not extract date from: {local_path}")
                continue
            
            file_parts = file.split('_')
            last = file_parts[-1].split('-')
            ed = int(last[0])
            if ed < 1 or ed > 52: 
                print(f"[ERROR] Invalid edition format in filename: {last}, {root}, {file}")
                edition = None
            elif len(file_parts) > 1 and len(last) > 1:
                edition = EDITIONS_MAPPINGS[ed]
            elif len(last)==1:
                edition = 'a'
            else: 
                print('no edition found')
            latest_rows.append({'alias': alias, 'date': date, 'edition': edition, 'local_path': strip_impresso_root(root)})

    if not found_mets_in_root:
        print(f"[WARNING] No METS file found in: {root}")
        
print("Total METS files found in LATEST BNL batch:", mets_count)

[WARNING] No METS file found in: /mnt/project_impresso/original/BNL-new/2025-impresso-3/
[WARNING] No METS file found in: /mnt/project_impresso/original/BNL-new/2025-impresso-3/freihet
Extracted date from: 1946-10-01_01-mets.xml
Extracted date from: 1945-10-19_01-mets.xml
Extracted date from: 1946-11-01_01-mets.xml
Extracted date from: 1945-07-27_01-mets.xml
Extracted date from: 1946-08-01_01-mets.xml
Extracted date from: 1945-09-22_01-mets.xml
Extracted date from: 1945-05-19_01-mets.xml
Extracted date from: 1945-06-20_01-mets.xml
Extracted date from: 1946-12-01_01-mets.xml
Extracted date from: 1945-07-20_01-mets.xml
Extracted date from: 1945-10-06_01-mets.xml
Extracted date from: 1945-11-17_01-mets.xml
Extracted date from: 1945-09-15_01-mets.xml
Extracted date from: 1945-08-25_01-mets.xml
Extracted date from: 1945-07-02_01-mets.xml
Extracted date from: 1945-05-04_01-mets.xml
Extracted date from: 1945-07-14_01-mets.xml
Extracted date from: 1945-08-04_01-mets.xml
Extracted date from: 19

In [17]:
latest_rows[:5]

[{'alias': 'freihet',
  'date': datetime.date(1946, 10, 1),
  'edition': 'a',
  'local_path': '/mnt/project_impresso/original/BNL-new/2025-impresso-3/freihet/70795_mfz29jpd1k'},
 {'alias': 'freihet',
  'date': datetime.date(1945, 10, 19),
  'edition': 'a',
  'local_path': '/mnt/project_impresso/original/BNL-new/2025-impresso-3/freihet/70795_t40w3g1h2h'},
 {'alias': 'freihet',
  'date': datetime.date(1946, 11, 1),
  'edition': 'a',
  'local_path': '/mnt/project_impresso/original/BNL-new/2025-impresso-3/freihet/70795_mndwhs39wg'},
 {'alias': 'freihet',
  'date': datetime.date(1945, 7, 27),
  'edition': 'a',
  'local_path': '/mnt/project_impresso/original/BNL-new/2025-impresso-3/freihet/70795_z4whbv3t7z'},
 {'alias': 'freihet',
  'date': datetime.date(1946, 8, 1),
  'edition': 'a',
  'local_path': '/mnt/project_impresso/original/BNL-new/2025-impresso-3/freihet/70795_bc7fzj60n4'}]

In [16]:
len(latest_rows)

83238

Number of issues found: 83266 (corresponding to number of mets files found). - 28 because of their mets file. It's themecht issues from 2015 to 2022

```
[ERROR] Invalid date format in filename: 1935-02-31_01-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/charribarri/70795_rn251rrk8k/1935-02-31_01-mets.xml
```

````
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_97qgm4mkbc/97qgm4mkbc-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_rv6cz4m1wb/rv6cz4m1wb-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_zcd68bfvt/zcd68bfvt-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_tmrf7bhs9j/tmrf7bhs9j-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_z7n095xs7n/z7n095xs7n-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_7th060nkbt/7th060nkbt-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_7b4hrgsmx6/7b4hrgsmx6-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_k6n2r7bt4z/k6n2r7bt4z-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_zbprm9hfpx/zbprm9hfpx-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_sphxckpj39/sphxckpj39-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_70js6zh2k2/70js6zh2k2-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_x2wx3xm63j/x2wx3xm63j-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_8rgsb1tmv/8rgsb1tmv-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_3689x8fcmg/3689x8fcmg-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_8mcthmsr6t/8mcthmsr6t-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_20tk49xgm/20tk49xgm-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_whjtk48ff/whjtk48ff-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_s0mdp9fqkt/s0mdp9fqkt-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_pvtz2jkf5/pvtz2jkf5-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_v8238jw3k/v8238jw3k-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_20p821s1hr/20p821s1hr-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_8047nmr7m/8047nmr7m-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_p5xtwmw4n/p5xtwmw4n-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_nbzg9g97r/nbzg9g97r-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_3t8z5k9mb/3t8z5k9mb-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_xtjgdkcmwz/xtjgdkcmwz-mets.xml
[ERROR] Could not extract date from: /mnt/project_impresso/original/BNL-new/2025-impresso-3/themecht/70795_ng68h0p39/ng68h0p39-mets.xml



alias themecht, from 2015 to 2022

In [18]:
final_rows = rows + new_rows + latest_rows
df_bnl = pd.DataFrame(final_rows)

In [19]:
df_bnl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 212941 entries, 0 to 212940
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   alias       212941 non-null  object
 1   date        212941 non-null  object
 2   edition     212941 non-null  object
 3   local_path  212941 non-null  object
dtypes: object(4)
memory usage: 6.5+ MB


### Remove the datetime object

In [40]:
for row in rows:
    row['date'] = row['date'].isoformat()

for row in new_rows:
    row['date'] = row['date'].isoformat()
    
for row in latest_rows:
    row['date'] = row['date'].isoformat()

In [41]:
len(rows), len(new_rows), len(latest_rows)

(97888, 31815, 83238)

### Add issues that are exceptions, 30 new issues

Luxwort 1898-04-29_01 a '/mnt/project_impresso/original/BNL/protected_019/1133553_Newspaper'

In [ ]:
exception1_row = []
exception1_row.append({'alias': 'luxwort', 'date': '1898-04-29', 'edition': 'a', 'local_path': '/BNL/protected_019/1133553_Newspaper'})

themecht: do list of ark_id (folder names) that are missing.

Maybe do a cas par cas

In [ ]:
exceptions_themecht = []
exceptions_themecht.extend([
    {'alias': 'themecht', 'date': '2021-10-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_97qgm4mkbc'},
    {'alias': 'themecht', 'date': '2019-04-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_rv6cz4m1wb'},
    {'alias': 'themecht', 'date': '2017-07-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_zcd68bfvt'},
    {'alias': 'themecht', 'date': '2021-04-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_tmrf7bhs9j'},
    {'alias': 'themecht', 'date': '2019-07-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_z7n095xs7n'},
    {'alias': 'themecht', 'date': '2021-01-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_7th060nkbt'},
    {'alias': 'themecht', 'date': '2021-07-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_7b4hrgsmx6'},
    {'alias': 'themecht', 'date': '2022-10-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_k6n2r7bt4z'},
    {'alias': 'themecht', 'date': '2022-04-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_zbprm9hfpx'},
    {'alias': 'themecht', 'date': '2020-07-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_sphxckpj39'},
    {'alias': 'themecht', 'date': '2020-01-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_70js6zh2k2'},
    {'alias': 'themecht', 'date': '2019-10-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_x2wx3xm63j'},
    {'alias': 'themecht', 'date': '2018-01-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_8rgsb1tmv'},
    {'alias': 'themecht', 'date': '2022-01-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_3689x8fcmg'},
    {'alias': 'themecht', 'date': '2020-04-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_8mcthmsr6t'},
    {'alias': 'themecht', 'date': '2018-04-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_20tk49xgm'},
    {'alias': 'themecht', 'date': '2016-01-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_whjtk48ff'},
    {'alias': 'themecht', 'date': '2020-10-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_s0mdp9fqkt'},
    {'alias': 'themecht', 'date': '2018-10-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_pvtz2jkf5'},
    {'alias': 'themecht', 'date': '2018-07-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_v8238jw3k'},
    {'alias': 'themecht', 'date': '2019-01-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_20p821s1hr'},
    {'alias': 'themecht', 'date': '2017-04-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_8047nmr7m'},
    {'alias': 'themecht', 'date': '2016-04-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_p5xtwmw4n'},
    {'alias': 'themecht', 'date': '2016-10-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_nbzg9g97r'},
    {'alias': 'themecht', 'date': '2016-07-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_3t8z5k9mb'},
    {'alias': 'themecht', 'date': '2022-07-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_xtjgdkcmwz'},
    {'alias': 'themecht', 'date': '2017-01-01', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/themecht/70795_ng68h0p39'}
])

charribarri 1935-02-31 -> put last day of février, the 28th

In [ ]:
exception2_row = []
exception2_row.append({'alias': 'charribarri', 'date': '1935-02-28', 'edition': 'a', 'local_path': '/BNL-new/2025-impresso-3/charribarri/70795_rn251rrk8k'})

### Create new DataFrame with all issues, even weird ones

In [65]:
complete_rows = rows + new_rows + latest_rows + exception1_row + exception2_row + exceptions_themecht
df_bnl_complete = pd.DataFrame(complete_rows)

In [66]:
df_bnl_complete.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 212970 entries, 0 to 212969
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   alias       212970 non-null  object
 1   date        212970 non-null  object
 2   edition     212970 non-null  object
 3   local_path  212970 non-null  object
dtypes: object(4)
memory usage: 6.5+ MB


### Duplicated issues

In [67]:
# check duplicated lines over subset 'alias', 'date', and 'edition'
duplicated_issues = df_bnl_complete[df_bnl_complete.duplicated(subset=['alias', 'date', 'edition'])]


In [68]:
duplicated_issues.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25353 entries, 97922 to 212939
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   alias       25353 non-null  object
 1   date        25353 non-null  object
 2   edition     25353 non-null  object
 3   local_path  25353 non-null  object
dtypes: object(4)
memory usage: 990.4+ KB


In [69]:
list(duplicated_issues['alias'].unique())

['luxzeit1858',
 'obermosel',
 'waechtersauer',
 'waeschfra',
 'avenirgdl',
 'gazgrdlux',
 'indeplux',
 'courriergdl',
 'volkfreu1869',
 'diekwochen',
 'luxwort',
 'luxzeit1844',
 'lunion',
 'socbotmem',
 'eschervolkszeitung',
 'jonghemecht',
 'natioun',
 'land1866',
 'handwerker',
 'wahrheit',
 'kommunistischegwerkschaftler',
 'kirchanzeiger1871',
 'gewerkschaftler',
 'socnatfauna',
 'fortschrit1870',
 'kulturkampf']

In [77]:
duplicated_issues[duplicated_issues['alias'] == 'socnatfauna']

,alias,date,edition,local_path
194350,socnatfauna,1893-09-01,a,/mnt/project_impresso/original/BNL-new/2025-im...
194351,socnatfauna,1892-10-01,a,/mnt/project_impresso/original/BNL-new/2025-im...
194352,socnatfauna,1906-02-28,a,/mnt/project_impresso/original/BNL-new/2025-im...
194353,socnatfauna,1893-07-01,a,/mnt/project_impresso/original/BNL-new/2025-im...
194354,socnatfauna,1906-03-31,a,/mnt/project_impresso/original/BNL-new/2025-im...
...,...,...,...,...
194506,socnatfauna,1897-03-31,a,/mnt/project_impresso/original/BNL-new/2025-im...
194507,socnatfauna,1897-07-31,a,/mnt/project_impresso/original/BNL-new/2025-im...
194508,socnatfauna,1901-12-31,a,/mnt/project_impresso/original/BNL-new/2025-im...
194509,socnatfauna,1905-04-30,a,/mnt/project_impresso/original/BNL-new/2025-im...


In [78]:
list(df_bnl_complete[(df_bnl_complete['alias'] == 'socnatfauna') & (df_bnl_complete['date'] == '1897-07-31')]['local_path'])

['/mnt/project_impresso/original/BNL/protected_044/2533293_serial_socnatfauna_1897-07-31_01',
 '/mnt/project_impresso/original/BNL-new/2025-impresso-3/socnatfauna/70795_tc9gxt']

### Solution for duplicated issue

Adding a new column 'to_ingest' with true or false, true for every issue that are not duplicated and false for the old copy of a duplicated issue 

if one issue duplicated on subset 'alias' 'date' and 'edition' if 'local_path' are  '/mnt/project_impresso/original/BNL/public_domain_007/2385868_newspaper_luxzeit1858_1859-01-06_01' and '/mnt/project_impresso/original/BNL-new/impresso/public_domain/newspaper/luxzeit1858/1859/z7mgb9' put false to the first one and true to the second one because it's the most recent or '/mnt/project_impresso/original/BNL/protected_044/2533293_serial_socnatfauna_1897-07-31_01' (false) and '/mnt/project_impresso/original/BNL-new/2025-impresso-3/socnatfauna/70795_tc9gxt' (true) 

-> BNL is the oldest, then BNL-new/impresso then BNL-new/2025-impresso-3

In [89]:
# 1) Assign priority based on folder
def compute_priority(path: str) -> int:
    if "BNL-new/2025-impresso-3" in path:
        return 3
    if "BNL-new/impresso" in path:
        return 2
    if "BNL" in path:
        return 1
    return 0  # fallback if unexpected path


In [90]:
df_bnl_complete_extended = df_bnl_complete.copy()

In [91]:
df_bnl_complete_extended.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 212970 entries, 0 to 212969
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   alias       212970 non-null  object
 1   date        212970 non-null  object
 2   edition     212970 non-null  object
 3   local_path  212970 non-null  object
dtypes: object(4)
memory usage: 6.5+ MB


In [92]:
df_bnl_complete_extended["priority"] = df_bnl_complete_extended["local_path"].apply(compute_priority)

In [93]:
# 2) For each duplicated issue (alias, date, edition):
#    mark only the highest-priority row as True
df_bnl_complete_extended["to_ingest"] = False  # default

# idxmax() returns index of the row with highest priority in each group
best_rows = df_bnl_complete_extended.groupby(["alias", "date", "edition"])["priority"].idxmax()

# Mark best rows as True
df_bnl_complete_extended.loc[best_rows, "to_ingest"] = True

# Optionally drop the helper column
df_bnl_complete_extended.drop(columns=["priority"], inplace=True)



In [94]:
df_bnl_complete_extended[df_bnl_complete_extended['to_ingest']]

,alias,date,edition,local_path,to_ingest
0,obermosel,1946-05-27,a,/mnt/project_impresso/original/BNL/protected_0...,True
1,obermosel,1940-04-11,a,/mnt/project_impresso/original/BNL/protected_0...,True
2,obermosel,1946-07-03,a,/mnt/project_impresso/original/BNL/protected_0...,True
3,obermosel,1946-05-04,a,/mnt/project_impresso/original/BNL/protected_0...,True
4,obermosel,1940-08-22,a,/mnt/project_impresso/original/BNL/protected_0...,True
...,...,...,...,...,...
212965,themecht,2016-04-01,a,/mnt/project_impresso/original/BNL-new/2025-im...,True
212966,themecht,2016-10-01,a,/mnt/project_impresso/original/BNL-new/2025-im...,True
212967,themecht,2016-07-01,a,/mnt/project_impresso/original/BNL-new/2025-im...,True
212968,themecht,2022-07-01,a,/mnt/project_impresso/original/BNL-new/2025-im...,True


### Which issues have been resent and won't be ingested

In [95]:
# Filter rows we will NOT ingest
not_ingested_df = df_bnl_complete_extended[df_bnl_complete_extended["to_ingest"] == False]

# Create a list of dictionaries
not_ingested_list = not_ingested_df[["alias", "date", "edition", "local_path"]].to_dict("records")

# Show preview
not_ingested_list[:5]


[{'alias': 'indeplux',
  'date': '1878-07-24',
  'edition': 'a',
  'local_path': '/mnt/project_impresso/original/BNL/protected_044/3070864_newspaper_indeplux_1878-07-24_01'},
 {'alias': 'indeplux',
  'date': '1879-09-11',
  'edition': 'a',
  'local_path': '/mnt/project_impresso/original/BNL/protected_044/3078763_newspaper_indeplux_1879-09-11_01'},
 {'alias': 'socnatfauna',
  'date': '1897-03-31',
  'edition': 'a',
  'local_path': '/mnt/project_impresso/original/BNL/protected_044/2533480_serial_socnatfauna_1897-03-31_01'},
 {'alias': 'indeplux',
  'date': '1879-10-23',
  'edition': 'a',
  'local_path': '/mnt/project_impresso/original/BNL/protected_044/3079564_newspaper_indeplux_1879-10-23_01'},
 {'alias': 'indeplux',
  'date': '1878-04-16',
  'edition': 'a',
  'local_path': '/mnt/project_impresso/original/BNL/protected_044/3069470_newspaper_indeplux_1878-04-16_01'}]

In [110]:
clean_json = list_to_nested_json_sorted(not_ingested_list, include_to_ingest=False)

In [111]:
with open("/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BNL/resent_issues_not_ingested.json", "w", encoding="utf-8") as f:
    json.dump(clean_json, f, indent=2, ensure_ascii=False)


### Issues to ingest

In [113]:
# Filter dataframe to only the rows where to_ingest == True
to_ingest_list = df_bnl_complete_extended[df_bnl_complete_extended["to_ingest"] == True].to_dict("records")

# Convert to nested JSON, WITHOUT including the to_ingest field
ingest_json = list_to_nested_json_sorted(to_ingest_list, include_to_ingest=False)

# Save
with open("/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BNL/issues_to_ingest.json", "w", encoding="utf-8") as f:
    json.dump(ingest_json, f, indent=2, ensure_ascii=False)


### All issues

In [114]:
all_issues_list = df_bnl_complete_extended.to_dict("records")

full_json = list_to_nested_json_sorted(all_issues_list, include_to_ingest=True)

with open("/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BNL/all_issues_with_ingest_flag.json", "w", encoding="utf-8") as f:
    json.dump(full_json, f, indent=2, ensure_ascii=False)


### Json file

Structure
```
{
  "Alias1": [
    {"date": "1910-01-01", "local_path": "/path/..."},
    {"date": "1910-02-01", "local_path": "/path/..."}
  ],
  "Alias2": [
    {"date": "1920-05-10", "local_path": "/path/..."}
  ]
}
```


In [33]:
# Convert datetime.date → string
df_bnl["date"] = df_bnl["date"].astype(str)

# Group rows by alias and convert to desired structure
output_dict = {}

for alias, group in df_bnl.groupby("alias"):
    output_dict[alias] = group[["date", "local_path"]].to_dict(orient="records")


In [34]:
output_dict

{'actionfem': [{'date': '1936-05-15',
   'local_path': '/mnt/project_impresso/original/BNL/protected_034/1930756_newspaper_actionfem_1936-05-15_01'},
  {'date': '1928-11-15',
   'local_path': '/mnt/project_impresso/original/BNL/protected_034/1925317_newspaper_actionfem_1928-11-15_01'},
  {'date': '1939-03-15',
   'local_path': '/mnt/project_impresso/original/BNL/protected_034/1934055_newspaper_actionfem_1939-03-15_01'},
  {'date': '1933-03-15',
   'local_path': '/mnt/project_impresso/original/BNL/protected_034/1929253_newspaper_actionfem_1933-03-15_01'},
  {'date': '1939-02-15',
   'local_path': '/mnt/project_impresso/original/BNL/protected_034/1933408_newspaper_actionfem_1939-02-15_01'},
  {'date': '1937-02-15',
   'local_path': '/mnt/project_impresso/original/BNL/protected_034/1931851_newspaper_actionfem_1937-02-15_01'},
  {'date': '1931-01-15',
   'local_path': '/mnt/project_impresso/original/BNL/protected_034/1927603_newspaper_actionfem_1931-01-15_01'},
  {'date': '1938-12-15',
   

In [ ]:

# Save to JSON file
with open("bnl_metadata.json", "w", encoding="utf-8") as f:
    json.dump(output_dict, f, indent=2, ensure_ascii=False)

### Json with year -> month -> day

From json

ORGANISE ORDER BY YEAR AND THEN MONTH

From DataFrame

In [26]:
nested_json = df_to_nested_json(df_bnl)

In [28]:
with open("/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BNL/bnl_metadata_organised.json", "w") as fp:
    json.dump(nested_json, fp, indent=2)


### TESTS

## Strip impresso root


In [8]:
import json

INPUT_JSON = "/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BNL/issues_to_ingest.json"
OUTPUT_JSON = "/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BNL/issues_to_ingest_stripped.json"

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

# Traverse and modify in place
for alias, years in data.items():
    for year, months in years.items():
        for month, issues in months.items():
            for issue in issues:
                if "local_path" in issue:
                    issue["local_path"] = strip_impresso_root(issue["local_path"])

# Dump updated JSON
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)


In [9]:
INPUT_JSON = "/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BNL/all_issues_with_ingest_flag.json"
OUTPUT_JSON = "/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BNL/all_issues_with_ingest_flag_stripped.json"

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

# Traverse and modify in place
for alias, years in data.items():
    for year, months in years.items():
        for month, issues in months.items():
            for issue in issues:
                if "local_path" in issue:
                    issue["local_path"] = strip_impresso_root(issue["local_path"])

# Dump updated JSON
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)
